# Premier League 2026/27 — Elite Article 13 analysis

This notebook combines the two executed source analyses behind **The Premier League Is Not a Forecast. It Is a Distribution.**

Source workspace: `daily_data_analytics_september2026`  
Snapshot date: 21 August 2026

The Elite repository intentionally does not duplicate the source data snapshots. To reproduce the analysis, run from the source workspace or update `DATA` and `OUT` paths to the corresponding snapshot and output directories.

# Premier League 2026/27: three genuinely different forecast views

**Snapshot date: 21 August 2026**

This notebook replaces the old collection of near-identical forecasts. It asks three different questions:

1. **Five-season history:** what do results across the last five seasons say, with recent seasons weighted more heavily and Championship results explicitly discounted?
2. **Current squad:** what does the present-day player layer say, using a 100-player worldwide EA FC reference, current FPL squads/prices, and five seasons of player production?
3. **Context-adjusted synthesis:** what happens when history and squad evidence are combined, then uncertainty is widened for managerial change, transfers, promotion and European workload?

The third view is the preferred forecast, but it is labelled a **synthesis**, not a falsely independent model. All title probabilities are simulation outputs, not statements of certainty.

Data references: [EA SPORTS FC 26 ratings](https://www.ea.com/games/ea-sports-fc/ratings?gender=0&orderBy=rank&page=1), [Fantasy Premier League API](https://fantasy.premierleague.com/api/bootstrap-static/), [football-data.co.uk](https://www.football-data.co.uk/englandm.php), and the [Premier League 2026/27 club list](https://www.premierleague.com/en/news/4673099/the-202627-premier-league-season-officially-starts/).

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, HTML

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
sns.set_theme(style="whitegrid", context="notebook")
COLORS = {"History": "#5877A3", "Squad": "#D7943B", "Context synthesis": "#5B9A78"}

ROOT = Path.cwd()
DATA = ROOT / "data" / "premier_league"
OUT = ROOT / "outputs"
OUT.mkdir(exist_ok=True)

players = pd.read_csv(DATA / "current_player_pool.csv")
player_history = pd.read_csv(DATA / "current_player_history.csv")
world100 = pd.read_csv(DATA / "ea_fc26_world_mens_top100.csv")

SEASONS = ["2021/22", "2022/23", "2023/24", "2024/25", "2025/26"]
MATCH_CODES = ["2122", "2223", "2324", "2425", "2526"]
match_frames = []
for season, code_ in zip(SEASONS, MATCH_CODES):
    for division in ("E0", "E1"):
        frame = pd.read_csv(DATA / f"matches_{code_}_{division}.csv")
        frame["season"] = season
        frame["division"] = division
        match_frames.append(frame)
matches = pd.concat(match_frames, ignore_index=True)

teams = sorted(players["current_team"].unique())
print(f"Loaded {len(players):,} current players across {len(teams)} clubs")
print(f"Worldwide EA reference: {len(world100):,} players")
print(f"Historical match rows: {len(matches):,} across five seasons and two divisions")

## 1. Build the player layer

The worldwide EA table is a **100-player calibration/reference set**, not a claim that all 100 play in England. The Premier League ranking is then built from the current FPL squads:

- **65% EA FC overall percentile:** a stable, cross-league quality anchor.
- **20% FPL price percentile within position:** a current expectation signal without comparing goalkeeper prices directly with forwards.
- **15% five-season FPL points-per-90 percentile within position:** recency-weighted production.

Players without Premier League history receive a neutral history percentile and lower reliability. They are not punished simply for arriving from another league. The final Premier League 100 has a transparent constraint: select each club's top three first, then fill the remaining 40 places by league-wide score.

In [ ]:
# Recency-weighted player history (five seasons, not last season alone)
season_weight = {"2021/22": 0.10, "2022/23": 0.15, "2023/24": 0.20,
                 "2024/25": 0.25, "2025/26": 0.30}
hist = player_history[player_history["season"].isin(SEASONS)].copy()
hist["weight"] = hist["season"].map(season_weight)
hist["weighted_points"] = hist["total_points"] * hist["weight"]
hist["weighted_minutes"] = hist["minutes"] * hist["weight"]
hist5 = hist.groupby("fpl_id", as_index=False).agg(
    weighted_points=("weighted_points", "sum"),
    weighted_minutes=("weighted_minutes", "sum"),
    raw_five_year_minutes=("minutes", "sum"),
)
hist5["points_per_90"] = np.where(
    hist5["weighted_minutes"] > 0,
    90 * hist5["weighted_points"] / hist5["weighted_minutes"], np.nan
)
hist5["history_reliability"] = (hist5["weighted_minutes"] / 2_000).clip(0, 1)

ranked = players.merge(hist5, on="fpl_id", how="left")
ranked["ea_percentile"] = ranked["ea_overall"].rank(pct=True)
ranked["price_percentile"] = ranked.groupby("fpl_position")["fpl_cost"].rank(pct=True)
ranked["history_percentile"] = ranked.groupby("fpl_position")["points_per_90"].rank(pct=True)
ranked["ea_percentile"] = ranked["ea_percentile"].fillna(0.45)
ranked["history_percentile"] = ranked["history_percentile"].fillna(0.50)
ranked["history_reliability"] = ranked["history_reliability"].fillna(0)
ranked["player_score"] = 100 * (
    0.65 * ranked["ea_percentile"]
    + 0.20 * ranked["price_percentile"]
    + 0.15 * ranked["history_percentile"]
)
ranked["unrestricted_rank"] = ranked["player_score"].rank(method="first", ascending=False).astype(int)

# Hard club floor: top three from every club, then best remaining players.
floor_ids = set(
    ranked.sort_values("player_score", ascending=False)
          .groupby("current_team", sort=False).head(3)["fpl_id"]
)
ordered_ids = ranked.sort_values("player_score", ascending=False)["fpl_id"].tolist()
selected_ids = set(floor_ids)
for player_id in ordered_ids:
    if len(selected_ids) >= 100:
        break
    selected_ids.add(player_id)

pl100 = ranked[ranked["fpl_id"].isin(selected_ids)].copy()
pl100 = pl100.sort_values("player_score", ascending=False).reset_index(drop=True)
pl100["PL_rank"] = np.arange(1, len(pl100) + 1)
unrestricted_ids = set(ranked.nsmallest(100, "unrestricted_rank")["fpl_id"])
pl100["selection_reason"] = np.where(
    pl100["fpl_id"].isin(unrestricted_ids), "merit top 100", "three-per-club floor"
)

club_counts = pl100.groupby("current_team").size().rename("players_in_PL100").sort_values(ascending=False)
assert len(world100) >= 100, "Worldwide reference contains fewer than 100 players"
assert len(pl100) == 100, "Premier League ranking must contain exactly 100 players"
assert club_counts.min() >= 3, "Every Premier League club must contribute at least three players"

pl100.to_csv(OUT / "premier_league_player_top100.csv", index=False)
validation = pd.DataFrame({
    "test": ["Worldwide reference", "Current player pool", "Premier League top 100", "Smallest club representation"],
    "value": [len(world100), len(players), len(pl100), int(club_counts.min())],
    "required": [">= 100", ">= 60", "= 100", ">= 3"],
    "status": ["PASS", "PASS", "PASS", "PASS"],
})
display(validation)

In [ ]:
# Worldwide reference and the full constrained Premier League ranking
world_display = world100[["reference_rank", "name", "overall", "position", "team", "league"]].copy()
world_display.columns = ["Rank", "Player", "EA", "Pos", "EA club", "League"]
pl_display = pl100[["PL_rank", "name", "current_team", "fpl_position", "ea_overall",
                    "fpl_cost", "points_per_90", "player_score", "unrestricted_rank", "selection_reason"]].copy()
pl_display.columns = ["Rank", "Player", "Current club", "Pos", "EA", "FPL £m", "5y Pts/90",
                      "Score", "Raw rank", "Selection"]

print("EA FC 26 WORLDWIDE MEN'S TOP 100 REFERENCE")
display(world_display.style.format({"EA": "{:.0f}"}).hide(axis="index"))
print("CONSTRAINED PREMIER LEAGUE TOP 100 — minimum three per club")
display(pl_display.style.format({"EA": "{:.0f}", "FPL £m": "{:.1f}", "5y Pts/90": "{:.2f}", "Score": "{:.1f}"}, na_rep="—").hide(axis="index"))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 7), gridspec_kw={"width_ratios": [1.2, 1]})
league_counts = world100["league"].value_counts().head(10).sort_values()
axes[0].barh(league_counts.index, league_counts.values, color="#5877A3")
axes[0].set_title("Worldwide top 100: leading leagues")
axes[0].set_xlabel("Players in EA worldwide reference")

count_plot = club_counts.sort_values()
colors = ["#D7943B" if v == 3 else "#5B9A78" for v in count_plot.values]
axes[1].barh(count_plot.index, count_plot.values, color=colors)
axes[1].axvline(3, color="#A23E48", linestyle="--", linewidth=1.5, label="hard minimum")
axes[1].set_title("Every Premier League club is represented")
axes[1].set_xlabel("Players in constrained PL top 100")
axes[1].legend(frameon=False)
plt.tight_layout()
plt.show()

### Transfer sanity check

The **current club** always comes from the 2026/27 FPL squad snapshot. The separate EA club column is deliberately retained because a mismatch identifies a transfer or a lag in the ratings database. That means Robertson at Tottenham, Henderson at Chelsea and Isak at Liverpool are assigned to their current clubs even if EA's source club has not caught up.

In [ ]:
named_checks = ["Andrew Robertson", "Jordan Henderson", "Alexander Isak", "Virgil van Dijk",
                "Victor Munoz", "Jeremy Jacquet", "Ronald Araujo"]
transfer_check = ranked[ranked["name"].isin(named_checks)][
    ["name", "current_team", "ea_overall", "ea_source_team", "fpl_cost", "player_score", "unrestricted_rank"]
].sort_values("current_team")
display(transfer_check.style.format({"ea_overall": "{:.0f}", "fpl_cost": "{:.1f}",
                                     "player_score": "{:.1f}"}, na_rep="—").hide(axis="index"))

## 2. Five-season club history

Each club-season is scored from points per game (65%) and goal difference per game (35%), standardised within that season and division. Championship strength is shrunk and shifted down before it is compared with Premier League strength. This is intentionally conservative: promotion is evidence of Championship excellence, not evidence that a club was already an above-average Premier League side.

The horizon comparison below is important. It shows exactly how much a last-season-only story changes when three and five years are included.

In [ ]:
def build_team_seasons(matches_):
    rows = []
    for (season, division), frame in matches_.groupby(["season", "division"]):
        clubs = sorted(set(frame["HomeTeam"]) | set(frame["AwayTeam"]))
        for club in clubs:
            home = frame[frame["HomeTeam"] == club]
            away = frame[frame["AwayTeam"] == club]
            gf = home["FTHG"].sum() + away["FTAG"].sum()
            ga = home["FTAG"].sum() + away["FTHG"].sum()
            points = (3 * (home["FTR"] == "H").sum() + (home["FTR"] == "D").sum()
                      + 3 * (away["FTR"] == "A").sum() + (away["FTR"] == "D").sum())
            played = len(home) + len(away)
            rows.append({"season": season, "division": division, "team_history": club,
                         "played": played, "points": points, "ppg": points / played,
                         "gd_per_game": (gf - ga) / played})
    out = pd.DataFrame(rows)
    for col in ["ppg", "gd_per_game"]:
        out[f"z_{col}"] = out.groupby(["season", "division"])[col].transform(
            lambda x: (x - x.mean()) / x.std(ddof=0)
        )
    out["within_division_strength"] = 0.65 * out["z_ppg"] + 0.35 * out["z_gd_per_game"]
    out["tier_adjusted_strength"] = np.where(
        out["division"].eq("E0"), out["within_division_strength"],
        0.55 * out["within_division_strength"] - 1.15
    )
    return out

team_seasons = build_team_seasons(matches)
history_name = {
    "Arsenal": "Arsenal", "Aston Villa": "Aston Villa", "Bournemouth": "Bournemouth",
    "Brentford": "Brentford", "Brighton": "Brighton", "Chelsea": "Chelsea",
    "Coventry City": "Coventry", "Crystal Palace": "Crystal Palace", "Everton": "Everton",
    "Fulham": "Fulham", "Hull City": "Hull", "Ipswich Town": "Ipswich", "Leeds": "Leeds",
    "Liverpool": "Liverpool", "Man City": "Man City", "Man Utd": "Man United",
    "Newcastle": "Newcastle", "Nott'm Forest": "Nott'm Forest", "Spurs": "Tottenham",
    "Sunderland": "Sunderland",
}
recency = dict(zip(SEASONS, [0.10, 0.15, 0.20, 0.25, 0.30]))

def horizon_table(n_seasons):
    chosen = SEASONS[-n_seasons:]
    weights = {s: recency[s] for s in chosen}
    total_w = sum(weights.values())
    records = []
    for club in teams:
        sub = team_seasons[(team_seasons["team_history"] == history_name[club]) &
                           (team_seasons["season"].isin(chosen))].copy()
        sub["w"] = sub["season"].map(weights)
        strength = (sub["tier_adjusted_strength"] * sub["w"]).sum() / total_w
        records.append({"current_team": club, f"history_{n_seasons}y": strength})
    result = pd.DataFrame(records)
    result[f"rank_{n_seasons}y"] = result[f"history_{n_seasons}y"].rank(ascending=False).astype(int)
    return result

history_compare = horizon_table(1).merge(horizon_table(3), on="current_team").merge(horizon_table(5), on="current_team")
history_compare = history_compare.sort_values("rank_5y")
history_compare.to_csv(OUT / "history_horizon_comparison.csv", index=False)
display(history_compare[["current_team", "rank_1y", "rank_3y", "rank_5y",
                         "history_1y", "history_3y", "history_5y"]]
        .style.background_gradient(subset=["history_1y", "history_3y", "history_5y"], cmap="RdYlGn")
        .format({"history_1y": "{:.2f}", "history_3y": "{:.2f}", "history_5y": "{:.2f}"})
        .hide(axis="index"))

In [ ]:
rank_heat = history_compare.set_index("current_team")[["rank_1y", "rank_3y", "rank_5y"]]
rank_heat.columns = ["Last season", "Past 3 seasons", "Past 5 seasons"]
rank_heat = rank_heat.sort_values("Past 5 seasons")
plt.figure(figsize=(8, 9))
sns.heatmap(rank_heat, annot=True, fmt=".0f", cmap="YlGnBu_r", cbar_kws={"label": "Rank"}, linewidths=.5)
plt.title("Changing the horizon changes the story")
plt.xlabel("")
plt.ylabel("")
plt.tight_layout()
plt.show()

## 3. Current squad strength and uncertainty

Team quality is calculated from the likely core (top 11), depth (players 12–15), and elite ceiling (top three). This avoids making one superstar equivalent to a complete squad. A source-club mismatch is used as a transparent proxy for transfer churn; missing EA coverage and weak historical coverage increase uncertainty.

Context is deliberately asymmetric: a new manager or a busy transfer window does **not** automatically make a club worse, but it does make the outcome wider. European competition adds a very small mean schedule penalty and a larger variance term. The promoted-club adjustment is already mainly carried by the tier-discounted history model.

In [ ]:
def normalise_club_name(value):
    aliases = {
        "Manchester City": "Man City", "Manchester United": "Man Utd",
        "Tottenham Hotspur": "Spurs", "Newcastle United": "Newcastle",
        "Newcastle Utd": "Newcastle", "Leeds United": "Leeds",
        "Nottingham Forest": "Nott'm Forest", "AFC Bournemouth": "Bournemouth",
        "Coventry": "Coventry City", "Hull": "Hull City", "Ipswich": "Ipswich Town",
    }
    return aliases.get(value, value)

ranked["ea_club_normalised"] = ranked["ea_source_team"].map(normalise_club_name)
squad_rows = []
for club, sub in ranked.groupby("current_team"):
    sub = sub.sort_values("player_score", ascending=False).reset_index(drop=True)
    # A valid XI prevents one club's unusually productive defenders (or forwards)
    # from occupying every notional starting place.
    formation = {"GK": 1, "DEF": 4, "MID": 4, "FWD": 2}
    core = pd.concat([
        sub[sub["fpl_position"] == position].head(count)
        for position, count in formation.items()
    ]).sort_values("player_score", ascending=False)
    remaining = sub[~sub["fpl_id"].isin(core["fpl_id"])]
    depth = remaining.head(4)
    top3 = sub.head(3)
    top15 = pd.concat([core, depth]).drop_duplicates("fpl_id")
    squad_rows.append({
        "current_team": club,
        "core_score": core["player_score"].mean(),
        "depth_score": depth["player_score"].mean(),
        "elite_score": top3["player_score"].mean(),
        "squad_raw": 0.70 * core["player_score"].mean() + 0.20 * depth["player_score"].mean()
                     + 0.10 * top3["player_score"].mean(),
        "transfer_share": (top15["ea_club_normalised"] != club).fillna(True).mean(),
        "missing_ea_share": top15["ea_overall"].isna().mean(),
        "history_reliability": top15["history_reliability"].mean(),
        "world_top100_anchors": top15["ea_global_rank"].notna().sum(),
    })
squads = pd.DataFrame(squad_rows)
squads["squad_strength"] = (squads["squad_raw"] - squads["squad_raw"].mean()) / squads["squad_raw"].std(ddof=0)

managers = {
    "Arsenal": ("Mikel Arteta", False), "Aston Villa": ("Unai Emery", False),
    "Bournemouth": ("Marco Rose", True), "Brentford": ("Keith Andrews", False),
    "Brighton": ("Fabian Hurzeler", False), "Chelsea": ("Xabi Alonso", True),
    "Coventry City": ("Frank Lampard", False), "Crystal Palace": ("Pierre Sage", True),
    "Everton": ("David Moyes", False), "Fulham": ("Alvaro Arbeloa", True),
    "Hull City": ("Sergej Jakirovic", False), "Ipswich Town": ("Gary O'Neil", True),
    "Leeds": ("Daniel Farke", False), "Liverpool": ("Andoni Iraola", True),
    "Man City": ("Enzo Maresca", True), "Man Utd": ("Michael Carrick", True),
    "Newcastle": ("Vacant at snapshot", True), "Nott'm Forest": ("Oliver Glasner", True),
    "Sunderland": ("Regis Le Bris", False), "Spurs": ("Roberto De Zerbi", True),
}
europe = {
    "Arsenal": "UCL", "Aston Villa": "UCL", "Liverpool": "UCL", "Man City": "UCL", "Man Utd": "UCL",
    "Bournemouth": "UEL", "Sunderland": "UEL", "Crystal Palace": "UEL", "Brighton": "UECL qualifier",
}
promoted = {"Coventry City", "Hull City", "Ipswich Town"}
second_year = {"Leeds", "Sunderland"}

context = pd.DataFrame({"current_team": teams})
context["manager"] = context["current_team"].map(lambda x: managers[x][0])
context["manager_change"] = context["current_team"].map(lambda x: managers[x][1])
context["europe"] = context["current_team"].map(europe).fillna("None")
context["competitions"] = np.where(context["europe"].eq("None"), 3, 4)
context["promoted"] = context["current_team"].isin(promoted)
context["second_year_PL"] = context["current_team"].isin(second_year)

team_model = (history_compare[["current_team", "history_5y"]]
              .merge(squads, on="current_team").merge(context, on="current_team"))
team_model["history_strength"] = ((team_model["history_5y"] - team_model["history_5y"].mean()) /
                                  team_model["history_5y"].std(ddof=0))
team_model["context_mean"] = 0.45 * team_model["history_strength"] + 0.55 * team_model["squad_strength"]
team_model["schedule_penalty"] = team_model["europe"].map(
    {"UCL": -0.06, "UEL": -0.05, "UECL qualifier": -0.04, "None": 0.0}
)
team_model["context_mean"] += team_model["schedule_penalty"]
team_model["context_strength"] = ((team_model["context_mean"] - team_model["context_mean"].mean()) /
                                  team_model["context_mean"].std(ddof=0))

# Team-specific season-level uncertainty. Manager/churn mostly widen forecasts rather than moving the mean.
season_vol = team_seasons[team_seasons["team_history"].isin(history_name.values())].groupby("team_history")["tier_adjusted_strength"].std()
team_model["historical_volatility"] = team_model["current_team"].map(lambda x: season_vol.get(history_name[x], 0.7)).fillna(0.7)
team_model["history_sigma"] = 0.38 + 0.10 * team_model["historical_volatility"].clip(0, 1.5)
team_model["squad_sigma"] = (0.36 + 0.18 * (1 - team_model["history_reliability"])
                             + 0.18 * team_model["transfer_share"] + 0.12 * team_model["missing_ea_share"])
team_model["context_sigma"] = (0.36 + 0.13 * team_model["manager_change"].astype(float)
                               + 0.18 * team_model["transfer_share"]
                               + 0.12 * team_model["promoted"].astype(float)
                               + 0.06 * team_model["second_year_PL"].astype(float)
                               + 0.04 * (team_model["competitions"] == 4).astype(float)
                               + 0.10 * team_model["missing_ea_share"])

team_model.to_csv(OUT / "team_model_inputs.csv", index=False)
display(team_model.sort_values("squad_strength", ascending=False)[
    ["current_team", "squad_strength", "core_score", "depth_score", "world_top100_anchors",
     "transfer_share", "history_reliability", "manager", "europe", "context_sigma"]
].style.format({"squad_strength": "{:.2f}", "core_score": "{:.1f}", "depth_score": "{:.1f}",
                "transfer_share": "{:.0%}", "history_reliability": "{:.0%}", "context_sigma": "{:.2f}"})
 .background_gradient(subset=["squad_strength"], cmap="RdYlGn").hide(axis="index"))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 8))
sq = team_model.sort_values("squad_strength")
axes[0].barh(sq["current_team"], sq["squad_strength"], color=np.where(sq["squad_strength"] >= 0, "#5B9A78", "#D7943B"))
axes[0].axvline(0, color="#333333", linewidth=1)
axes[0].set_title("Current squad layer")
axes[0].set_xlabel("Standardised squad strength")

unc = team_model.sort_values("context_sigma")
axes[1].barh(unc["current_team"], unc["context_sigma"], color="#8C6BB1")
axes[1].set_title("Context model: season-level uncertainty")
axes[1].set_xlabel("Latent strength standard deviation")
plt.tight_layout()
plt.show()

## 4. Simulate the league three ways

Every view uses the same double round-robin fixture structure and match-outcome engine. What changes is the evidence supplied to it:

- **History:** five-season club strength and historical volatility.
- **Squad:** present player layer and data-reliability/churn uncertainty.
- **Context synthesis:** 45% history + 55% squad, with manager, transfer, promotion and workload uncertainty.

Before each simulated season, each team receives a latent season shock. This is crucial: a simulation that fixes team strengths for all 38 matches manufactures excessive certainty. Position ties are broken by a tiny random jitter rather than alphabetical order.

In [ ]:
def simulate_league(strength, sigma, n_sims=8_000, seed=202627):
    rng = np.random.default_rng(seed)
    n_teams = len(teams)
    base = np.asarray(strength, dtype=float)
    spread = np.asarray(sigma, dtype=float)
    latent = base[None, :] + rng.normal(0, spread[None, :], size=(n_sims, n_teams))
    points = np.zeros((n_sims, n_teams), dtype=np.int16)

    for home in range(n_teams):
        for away in range(n_teams):
            if home == away:
                continue
            diff = latent[:, home] - latent[:, away]
            # Softmax logits: home advantage, strength gap, and a draw channel.
            home_logit = 0.22 + 0.35 * diff
            away_logit = -0.22 - 0.35 * diff
            draw_logit = -0.08 - 0.08 * np.abs(diff)
            max_logit = np.maximum.reduce([home_logit, away_logit, draw_logit])
            eh = np.exp(home_logit - max_logit)
            ea = np.exp(away_logit - max_logit)
            ed = np.exp(draw_logit - max_logit)
            total = eh + ea + ed
            p_home, p_draw = eh / total, ed / total
            u = rng.random(n_sims)
            home_win = u < p_home
            draw = (u >= p_home) & (u < p_home + p_draw)
            away_win = ~(home_win | draw)
            points[:, home] += 3 * home_win + draw
            points[:, away] += 3 * away_win + draw

    jitter = rng.normal(0, 1e-3, size=points.shape)
    order = np.argsort(-(points + jitter), axis=1)
    positions = np.empty_like(order)
    positions[np.arange(n_sims)[:, None], order] = np.arange(1, n_teams + 1)
    summary = pd.DataFrame({
        "current_team": teams,
        "mean_points": points.mean(axis=0),
        "p10_points": np.percentile(points, 10, axis=0),
        "p90_points": np.percentile(points, 90, axis=0),
        "title_probability": (positions == 1).mean(axis=0),
        "top4_probability": (positions <= 4).mean(axis=0),
        "relegation_probability": (positions >= 18).mean(axis=0),
        "median_position": np.median(positions, axis=0),
        "position_p10": np.percentile(positions, 10, axis=0),
        "position_p90": np.percentile(positions, 90, axis=0),
    })
    return summary, points, positions

model_specs = {
    "History": ("history_strength", "history_sigma", 1001),
    "Squad": ("squad_strength", "squad_sigma", 2002),
    "Context synthesis": ("context_strength", "context_sigma", 3003),
}
summaries, points_draws, position_draws = {}, {}, {}
ordered_model = team_model.set_index("current_team").loc[teams]
for model, (strength_col, sigma_col, seed) in model_specs.items():
    summaries[model], points_draws[model], position_draws[model] = simulate_league(
        ordered_model[strength_col], ordered_model[sigma_col], seed=seed
    )
    summaries[model]["model"] = model

all_forecasts = pd.concat(summaries.values(), ignore_index=True)
all_forecasts.to_csv(OUT / "three_view_forecasts.csv", index=False)
print("Completed", len(model_specs) * 8_000, "season simulations")

In [ ]:
# Side-by-side forecast plots: same engine, genuinely different inputs.
fig, axes = plt.subplots(1, 3, figsize=(19, 9), sharex=True)
for ax, model in zip(axes, model_specs):
    view = summaries[model].sort_values("mean_points")
    xerr = np.vstack([view["mean_points"] - view["p10_points"], view["p90_points"] - view["mean_points"]])
    ax.errorbar(view["mean_points"], view["current_team"], xerr=xerr, fmt="o",
                color=COLORS[model], ecolor="#B8B8B8", capsize=2, markersize=5)
    ax.set_title(model)
    ax.set_xlabel("Points: mean and 10–90% interval")
axes[0].set_ylabel("")
plt.suptitle("Three forecast views: disagreement is information", y=1.01, fontsize=15)
plt.tight_layout()
plt.show()

In [ ]:
comparison = None
for model, frame in summaries.items():
    small = frame[["current_team", "mean_points", "title_probability", "top4_probability", "relegation_probability"]].copy()
    small.columns = ["current_team"] + [f"{model}: {c}" for c in small.columns[1:]]
    comparison = small if comparison is None else comparison.merge(small, on="current_team")
comparison = comparison.sort_values("Context synthesis: mean_points", ascending=False)

title_cols = [f"{m}: title_probability" for m in model_specs]
points_cols = [f"{m}: mean_points" for m in model_specs]
table_view = comparison[["current_team"] + points_cols + title_cols].copy()
display(table_view.style
        .format({**{c: "{:.1f}" for c in points_cols}, **{c: "{:.1%}" for c in title_cols}})
        .background_gradient(subset=title_cols, cmap="YlOrRd")
        .hide(axis="index"))

In [ ]:
preferred = summaries["Context synthesis"].sort_values("mean_points", ascending=False).copy()
preferred["points_interval"] = preferred.apply(lambda r: f"{r.p10_points:.0f}–{r.p90_points:.0f}", axis=1)
preferred_table = preferred[["current_team", "mean_points", "points_interval", "title_probability",
                             "top4_probability", "relegation_probability", "median_position",
                             "position_p10", "position_p90"]]
display(preferred_table.style.format({
    "mean_points": "{:.1f}", "title_probability": "{:.1%}", "top4_probability": "{:.1%}",
    "relegation_probability": "{:.1%}", "median_position": "{:.0f}",
    "position_p10": "{:.0f}", "position_p90": "{:.0f}",
}).background_gradient(subset=["title_probability"], cmap="YlOrRd").hide(axis="index"))

In [ ]:
# Full finishing-position distribution in the preferred view.
pos = position_draws["Context synthesis"]
pos_prob = np.vstack([(pos == rank).mean(axis=0) for rank in range(1, 21)]).T
order = preferred["current_team"].tolist()
order_idx = [teams.index(t) for t in order]
plt.figure(figsize=(15, 9))
sns.heatmap(pos_prob[order_idx], cmap="mako", xticklabels=range(1, 21), yticklabels=order,
            cbar_kws={"label": "Probability"})
plt.title("Preferred synthesis: every club's full finishing-position distribution")
plt.xlabel("Finishing position")
plt.ylabel("")
plt.tight_layout()
plt.show()

In [ ]:
# Diagnose where uncertainty is coming from, rather than hiding it in one number.
diagnostic = team_model[["current_team", "context_sigma", "manager_change", "transfer_share",
                         "promoted", "second_year_PL", "competitions", "missing_ea_share"]].copy()
diagnostic = diagnostic.merge(preferred[["current_team", "position_p10", "position_p90"]], on="current_team")
diagnostic["position_range_10_90"] = diagnostic["position_p90"] - diagnostic["position_p10"]
diagnostic = diagnostic.sort_values(["context_sigma", "position_range_10_90"], ascending=False)
display(diagnostic.style.format({"context_sigma": "{:.2f}", "transfer_share": "{:.0%}",
                                 "missing_ea_share": "{:.0%}", "position_range_10_90": "{:.0f}"})
        .background_gradient(subset=["context_sigma", "position_range_10_90"], cmap="Purples")
        .hide(axis="index"))

## Method review: what changed, and what still needs caution

### Why the old models converged

The previous approaches repeatedly transformed the same small set of hand-entered team beliefs. Averaging them reduced visible disagreement without adding independent information. Fixed team strength across all 38 games then converted small mean differences into implausibly confident title probabilities.

### What is better here

- History and squad evidence are now separate before synthesis.
- Five seasons replace last-season anchoring; one-, three- and five-year ranks remain visible.
- The player pool covers every current club, including promoted clubs and second-year Leeds/Sunderland.
- The worldwide reference contains 100 players, while the Premier League top 100 guarantees at least three per club and labels constraint-driven inclusions.
- Current FPL club assignment captures transfers; EA's old club is retained as a churn signal.
- Managerial change, transfers and workload chiefly widen variance rather than becoming arbitrary bonuses or penalties.
- Season-level shocks prevent the model from acting as though team strength is known perfectly in August.

### What this still is not

This is a transparent scenario model, not yet a calibrated betting model. The blend weights, Championship discount and uncertainty coefficients should be walk-forward backtested on earlier seasons. Injuries, minutes projections, tactics, transfer fees/wages and match-level expected-goal data would materially improve the squad layer. EA and FPL are useful priors, but neither is ground truth.

**Interpret the gaps between the three views and the width of each interval as part of the answer. A single league-table prediction is the least honest output here.**

---

# Part Two — three season engines

Part One builds the player, historical and squad evidence layers. Part Two holds those inputs fixed and changes the simulation methodology.

# Premier League 2026/27 — Part Two: three season engines

**Snapshot date: 21 August 2026**

Part One established the evidence base. Part Two holds that evidence fixed and changes the simulation methodology. This is the important test: disagreement should come from different assumptions about how football seasons unfold, not from three relabelled versions of the same formula.

1. **Model 1 — Structural mixture + K:** history and squad quality, with one global uncertainty control, $K$.
2. **Model 2 — Contextual match engine:** the same structural mean, but uncertainty is decomposed into stadium, coach, European workload, transfers, promotion and match availability.
3. **Model 3 — Nonlinear ML form engine:** a gradient-boosted match classifier trained chronologically on five seasons, driven by pre-match Elo, rolling form and venue form, then simulated with heavy-tailed season regimes.

The models share the same 20 clubs and double round-robin structure. Their outputs are probabilities, not promises.

In [ ]:
from pathlib import Path
from collections import defaultdict, deque
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
import seaborn as sns
from IPython.display import display, Markdown
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, log_loss

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
sns.set_theme(style="whitegrid", context="notebook")

ROOT = Path.cwd()
DATA = ROOT / "data" / "premier_league"
OUT = ROOT / "outputs"
OUT.mkdir(exist_ok=True)

team_inputs = pd.read_csv(OUT / "team_model_inputs.csv")
pl100 = pd.read_csv(OUT / "premier_league_player_top100.csv")
history_horizons = pd.read_csv(OUT / "history_horizon_comparison.csv")
teams = sorted(team_inputs["current_team"].tolist())
ordered = team_inputs.set_index("current_team").loc[teams]

SEASONS = ["2021/22", "2022/23", "2023/24", "2024/25", "2025/26"]
MATCH_CODES = ["2122", "2223", "2324", "2425", "2526"]
frames = []
for season, code_ in zip(SEASONS, MATCH_CODES):
    for division in ("E0", "E1"):
        frame = pd.read_csv(DATA / f"matches_{code_}_{division}.csv")
        frame["season"] = season
        frame["division"] = division
        frames.append(frame)
matches = pd.concat(frames, ignore_index=True)

history_name = {
    "Arsenal": "Arsenal", "Aston Villa": "Aston Villa", "Bournemouth": "Bournemouth",
    "Brentford": "Brentford", "Brighton": "Brighton", "Chelsea": "Chelsea",
    "Coventry City": "Coventry", "Crystal Palace": "Crystal Palace", "Everton": "Everton",
    "Fulham": "Fulham", "Hull City": "Hull", "Ipswich Town": "Ipswich", "Leeds": "Leeds",
    "Liverpool": "Liverpool", "Man City": "Man City", "Man Utd": "Man United",
    "Newcastle": "Newcastle", "Nott'm Forest": "Nott'm Forest", "Spurs": "Tottenham",
    "Sunderland": "Sunderland",
}

assert len(teams) == 20
assert len(pl100) == 100 and pl100.groupby("current_team").size().min() >= 3
print("Part One inputs validated:", len(teams), "clubs and", len(pl100), "ranked PL players")

## Shared scoring and summary functions

The match engine uses three outcome channels—home win, draw and away win. A pre-season team shock persists across the whole campaign, because a real surprise season is correlated across matches. Treating every game as an independent coin flip would understate season variance.

In [ ]:
def standardise(values):
    values = np.asarray(values, dtype=float)
    return (values - values.mean()) / values.std(ddof=0)

def finish_summary(points, positions, model_name):
    return pd.DataFrame({
        "current_team": teams,
        "model": model_name,
        "mean_points": points.mean(axis=0),
        "p10_points": np.percentile(points, 10, axis=0),
        "p90_points": np.percentile(points, 90, axis=0),
        "title_probability": (positions == 1).mean(axis=0),
        "top4_probability": (positions <= 4).mean(axis=0),
        "relegation_probability": (positions >= 18).mean(axis=0),
        "median_position": np.median(positions, axis=0),
        "position_p10": np.percentile(positions, 10, axis=0),
        "position_p90": np.percentile(positions, 90, axis=0),
    })

def positions_from_points(points, rng):
    jitter = rng.normal(0, 1e-3, size=points.shape)
    order_ = np.argsort(-(points + jitter), axis=1)
    positions = np.empty_like(order_)
    positions[np.arange(len(points))[:, None], order_] = np.arange(1, len(teams) + 1)
    return positions

def apply_softmax_fixture(points, latent, home, away, rng, home_bonus=0.22,
                          coefficient=0.35, home_match_noise=None, away_match_noise=None):
    diff = latent[:, home] - latent[:, away]
    if home_match_noise is not None:
        diff = diff + home_match_noise
    if away_match_noise is not None:
        diff = diff - away_match_noise
    h_logit = home_bonus + coefficient * diff
    a_logit = -home_bonus - coefficient * diff
    d_logit = -0.08 - 0.08 * np.abs(diff)
    ceiling = np.maximum.reduce([h_logit, a_logit, d_logit])
    eh, ea, ed = np.exp(h_logit-ceiling), np.exp(a_logit-ceiling), np.exp(d_logit-ceiling)
    denom = eh + ea + ed
    p_h, p_d = eh / denom, ed / denom
    u = rng.random(len(points))
    h_win = u < p_h
    draw = (u >= p_h) & (u < p_h + p_d)
    a_win = ~(h_win | draw)
    points[:, home] += 3*h_win + draw
    points[:, away] += 3*a_win + draw

## Model 1 — History/squad mixture with randomiser K

**Thesis:** durable club performance and present squad quality explain most of the mean; everything we do not know is represented by one season-level randomiser.

The mean is a 50/50 mix of five-season history and current squad quality. $K$ is the standard deviation of the persistent pre-season shock. Small $K$ says the ranking is well known; large $K$ allows genuine breakout and collapse seasons. The base case uses **K = 0.45**, while the sensitivity chart shows why certainty is a modelling choice.

In [ ]:
base_mean = standardise(0.50 * ordered["history_strength"] + 0.50 * ordered["squad_strength"])

def simulate_model_one(K=0.45, n_sims=8_000, seed=1101):
    rng = np.random.default_rng(seed)
    latent = base_mean[None, :] + rng.normal(0, K, size=(n_sims, len(teams)))
    points = np.zeros((n_sims, len(teams)), dtype=np.int16)
    for home in range(len(teams)):
        for away in range(len(teams)):
            if home != away:
                apply_softmax_fixture(points, latent, home, away, rng)
    positions = positions_from_points(points, rng)
    return finish_summary(points, positions, "Model 1: mixture + K"), points, positions

m1, points1, positions1 = simulate_model_one()
K_rows = []
for K, seed in [(0.20, 1201), (0.45, 1202), (0.75, 1203)]:
    result, _, _ = simulate_model_one(K=K, n_sims=4_000, seed=seed)
    result["K"] = K
    K_rows.append(result)
K_sensitivity = pd.concat(K_rows, ignore_index=True)

display(m1.sort_values("mean_points", ascending=False).style.format({
    "mean_points":"{:.1f}", "p10_points":"{:.0f}", "p90_points":"{:.0f}",
    "title_probability":"{:.1%}", "top4_probability":"{:.1%}",
    "relegation_probability":"{:.1%}", "median_position":"{:.0f}",
    "position_p10":"{:.0f}", "position_p90":"{:.0f}"}).hide(axis="index"))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
view = m1.sort_values("mean_points").tail(12)
xerr = np.vstack([view.mean_points-view.p10_points, view.p90_points-view.mean_points])
axes[0].errorbar(view.mean_points, view.current_team, xerr=xerr, fmt="o", color="#315A7D", ecolor="#AAB7C4", capsize=2)
axes[0].set_title("Model 1 base case: K = 0.45")
axes[0].set_xlabel("Points, mean and 10–90% interval")

leaders = m1.nlargest(6, "title_probability")["current_team"]
sens = K_sensitivity[K_sensitivity.current_team.isin(leaders)]
sns.lineplot(data=sens, x="K", y="title_probability", hue="current_team", marker="o", ax=axes[1])
axes[1].yaxis.set_major_formatter(lambda x, _: f"{x:.0%}")
axes[1].set_title("K controls how certain the answer looks")
axes[1].set_ylabel("Title probability")
axes[1].legend(title="", frameon=False)
plt.tight_layout(); plt.show()

## Model 2 — Contextual stadium, coach and workload engine

**Thesis:** the structural ranking is useful, but the path through a season is club-specific. Home advantage is not identical at every stadium; a new coach can produce improvement or disruption; European matches create intermittent fatigue; transfers and promotion widen the possible season.

Crucially, coach and transfer effects are centred near zero. The model admits that their direction is unknown instead of awarding arbitrary “new manager points”. European fatigue is applied intermittently at match level, not as a deterministic full-season punishment.

In [ ]:
# Recency-weighted stadium effect: home PPG minus away PPG, shrunk toward league average.
venue_rows = []
season_weights = dict(zip(SEASONS, [0.10, 0.15, 0.20, 0.25, 0.30]))
for club in teams:
    hist_club = history_name[club]
    pieces = []
    for season, frame in matches.groupby("season"):
        home = frame[frame.HomeTeam == hist_club]
        away = frame[frame.AwayTeam == hist_club]
        if len(home) and len(away):
            home_pts = 3*(home.FTR == "H").sum() + (home.FTR == "D").sum()
            away_pts = 3*(away.FTR == "A").sum() + (away.FTR == "D").sum()
            pieces.append((season_weights[season], home_pts/len(home), away_pts/len(away)))
    w = sum(x[0] for x in pieces)
    venue_rows.append({"current_team":club,
                       "home_ppg":sum(x[0]*x[1] for x in pieces)/w,
                       "away_ppg":sum(x[0]*x[2] for x in pieces)/w})
venues = pd.DataFrame(venue_rows)
venues["raw_home_lift"] = venues.home_ppg - venues.away_ppg
venues["stadium_z"] = standardise(venues.raw_home_lift)
venues["stadium_bonus"] = (0.07 * venues.stadium_z).clip(-0.11, 0.11)

context2 = ordered.reset_index().merge(venues, on="current_team").set_index("current_team").loc[teams]

def simulate_model_two(n_sims=8_000, seed=2202):
    rng = np.random.default_rng(seed)
    n = len(teams)
    # Explicit independent components, all persistent across the season.
    base_shock = rng.normal(0, 0.32, size=(n_sims, n))
    coach_sd = np.where(context2.manager_change, 0.20, 0.06)
    coach_shock = rng.normal(0, coach_sd[None, :], size=(n_sims, n))
    transfer_sd = 0.30 * context2.transfer_share.to_numpy()
    transfer_shock = rng.normal(0, transfer_sd[None, :], size=(n_sims, n))
    transition_sd = 0.16*context2.promoted.to_numpy() + 0.09*context2.second_year_PL.to_numpy()
    transition_shock = rng.standard_t(5, size=(n_sims, n)) * transition_sd[None, :]
    euro_flag = (context2.competitions.to_numpy() == 4)
    euro_season = rng.normal(-0.03*euro_flag, 0.07*euro_flag, size=(n_sims, n))
    latent = base_mean[None, :] + base_shock + coach_shock + transfer_shock + transition_shock + euro_season
    points = np.zeros((n_sims, n), dtype=np.int16)

    for home in range(n):
        for away in range(n):
            if home == away:
                continue
            # Availability affects every club; European fatigue occurs only in a subset of matches.
            home_noise = rng.normal(0, 0.08, n_sims)
            away_noise = rng.normal(0, 0.08, n_sims)
            if euro_flag[home]:
                home_noise -= rng.binomial(1, 0.28, n_sims) * np.abs(rng.normal(0.10, 0.04, n_sims))
            if euro_flag[away]:
                away_noise -= rng.binomial(1, 0.28, n_sims) * np.abs(rng.normal(0.10, 0.04, n_sims))
            apply_softmax_fixture(points, latent, home, away, rng,
                                  home_bonus=0.22 + context2.stadium_bonus.iloc[home],
                                  home_match_noise=home_noise, away_match_noise=away_noise)
    positions = positions_from_points(points, rng)
    components = pd.DataFrame({
        "current_team":teams, "stadium_bonus":context2.stadium_bonus.to_numpy(),
        "coach_sd":coach_sd, "transfer_sd":transfer_sd, "transition_sd":transition_sd,
        "european_competition":euro_flag,
    })
    return finish_summary(points, positions, "Model 2: contextual"), points, positions, components

m2, points2, positions2, context_components = simulate_model_two()
display(m2.sort_values("mean_points", ascending=False).style.format({
    "mean_points":"{:.1f}", "p10_points":"{:.0f}", "p90_points":"{:.0f}",
    "title_probability":"{:.1%}", "top4_probability":"{:.1%}",
    "relegation_probability":"{:.1%}", "median_position":"{:.0f}",
    "position_p10":"{:.0f}", "position_p90":"{:.0f}"}).hide(axis="index"))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 8))
view = m2.sort_values("mean_points").tail(12)
xerr = np.vstack([view.mean_points-view.p10_points, view.p90_points-view.mean_points])
axes[0].errorbar(view.mean_points, view.current_team, xerr=xerr, fmt="o", color="#9B5C24", ecolor="#D8B692", capsize=2)
axes[0].set_title("Model 2: context changes the width")
axes[0].set_xlabel("Points, mean and 10–90% interval")

drivers = context_components.set_index("current_team")[["coach_sd","transfer_sd","transition_sd"]]
drivers["europe_sd"] = context_components.set_index("current_team").european_competition * 0.07
drivers = drivers.assign(total=drivers.sum(axis=1)).sort_values("total").drop(columns="total").tail(12)
drivers.plot.barh(stacked=True, ax=axes[1], color=["#714C8C", "#477A6A", "#B77835", "#54759A"])
axes[1].set_title("Explicit season-shock components")
axes[1].set_xlabel("Standard-deviation contribution")
axes[1].legend(title="", frameon=False)
plt.tight_layout(); plt.show()

## Model 3 — Gradient-boosted nonlinear form engine

**Thesis:** match probabilities are nonlinear and path-dependent. A five-match change in form should not necessarily mean the same thing for a dominant side and a struggling promoted club.

This model creates a chronological, leakage-free pre-match training table from the last five seasons of Premier League and Championship results. Features are pre-match Elo, rolling points, rolling goal difference, home-only form, away-only form, rest difference and season progress. The 2025/26 season is held out for an honest validation score before the model is refitted on all five seasons.

For 2026/27, current squad quality adjusts the final historical Elo prior. A heavy-tailed regime shock admits rare breakout/collapse campaigns, while form and Elo update after every simulated match.

In [ ]:
FEATURES = ["elo_diff", "recent_ppg_diff", "recent_gd_diff", "home_venue_ppg",
            "away_venue_ppg", "rest_diff", "division_top", "season_progress"]

def initial_state(base_elo):
    return {"elo":base_elo, "ppg":deque(maxlen=10), "gd":deque(maxlen=10),
            "home_ppg":deque(maxlen=8), "away_ppg":deque(maxlen=8), "last_date":None}

def mean_or(values, default):
    return float(np.mean(values)) if len(values) else default

def make_ml_training(matches_):
    data = matches_.copy()
    data["parsed_date"] = pd.to_datetime(data.Date, dayfirst=True, errors="coerce")
    data = data.sort_values(["parsed_date", "division", "HomeTeam", "AwayTeam"]).reset_index(drop=True)
    states, rows = {}, []
    season_game = defaultdict(int)
    season_total = data.groupby(["season","division"]).size().to_dict()
    for row in data.itertuples():
        default_elo = 1500 if row.division == "E0" else 1380
        hs = states.setdefault(row.HomeTeam, initial_state(default_elo))
        ass = states.setdefault(row.AwayTeam, initial_state(default_elo))
        if hs["last_date"] is None or ass["last_date"] is None:
            rest_diff = 0
        else:
            rest_diff = np.clip((row.parsed_date-hs["last_date"]).days - (row.parsed_date-ass["last_date"]).days, -10, 10)
        key = (row.season, row.division)
        rows.append({
            "season":row.season, "division":row.division, "home_team":row.HomeTeam, "away_team":row.AwayTeam,
            "elo_diff":hs["elo"]-ass["elo"],
            "recent_ppg_diff":mean_or(hs["ppg"],1.35)-mean_or(ass["ppg"],1.35),
            "recent_gd_diff":mean_or(hs["gd"],0)-mean_or(ass["gd"],0),
            "home_venue_ppg":mean_or(hs["home_ppg"],1.55),
            "away_venue_ppg":mean_or(ass["away_ppg"],1.15),
            "rest_diff":rest_diff, "division_top":int(row.division=="E0"),
            "season_progress":season_game[key]/season_total[key], "target":row.FTR,
        })
        season_game[key] += 1
        hpts = 3 if row.FTR == "H" else 1 if row.FTR == "D" else 0
        apts = 3 if row.FTR == "A" else 1 if row.FTR == "D" else 0
        margin = row.FTHG-row.FTAG
        hs["ppg"].append(hpts); ass["ppg"].append(apts)
        hs["gd"].append(margin); ass["gd"].append(-margin)
        hs["home_ppg"].append(hpts); ass["away_ppg"].append(apts)
        expected = 1/(1+10**(-((hs["elo"]-ass["elo"]+65)/400)))
        actual = 1 if row.FTR == "H" else .5 if row.FTR == "D" else 0
        change = 22*np.log1p(abs(margin))*(actual-expected)
        hs["elo"] += change; ass["elo"] -= change
        hs["last_date"] = row.parsed_date; ass["last_date"] = row.parsed_date
    return pd.DataFrame(rows), states

ml_data, final_states = make_ml_training(matches)
ml_data.to_csv(OUT / "ml_training_table.csv", index=False)
train = ml_data[ml_data.season != "2025/26"]
test = ml_data[ml_data.season == "2025/26"]
ml_validation = HistGradientBoostingClassifier(
    learning_rate=.025, max_iter=250, max_leaf_nodes=5, min_samples_leaf=80,
    l2_regularization=5.0, random_state=3301
).fit(train[FEATURES], train.target)
test_probability = ml_validation.predict_proba(test[FEATURES])
test_prediction = ml_validation.predict(test[FEATURES])
onehot = pd.get_dummies(pd.Categorical(test.target, categories=ml_validation.classes_)).to_numpy()
class_prior = train.target.value_counts(normalize=True).reindex(ml_validation.classes_).to_numpy()
baseline_probability = np.tile(class_prior, (len(test), 1))
validation_metrics = pd.DataFrame({
    "metric":["Accuracy", "Multiclass log loss", "Multiclass Brier score"],
    "ML — 2025/26 holdout":[accuracy_score(test.target,test_prediction),
                            log_loss(test.target,test_probability,labels=ml_validation.classes_),
                            np.mean(np.sum((test_probability-onehot)**2,axis=1))],
    "Unconditional baseline":[class_prior.max(),
                              log_loss(test.target,baseline_probability,labels=ml_validation.classes_),
                              np.mean(np.sum((baseline_probability-onehot)**2,axis=1))],
})
display(validation_metrics.style.format({"ML — 2025/26 holdout":"{:.3f}",
                                         "Unconditional baseline":"{:.3f}"}).hide(axis="index"))

ml_model = HistGradientBoostingClassifier(
    learning_rate=.025, max_iter=250, max_leaf_nodes=5, min_samples_leaf=80,
    l2_regularization=5.0, random_state=3302
).fit(ml_data[FEATURES], ml_data.target)
print("Training rows:", len(ml_data), "| holdout rows:", len(test), "| classes:", list(ml_model.classes_))

In [ ]:
def circle_schedule(n):
    fixed = list(range(n))
    first_half = []
    for rnd in range(n-1):
        pairs = []
        for i in range(n//2):
            a, b = fixed[i], fixed[n-1-i]
            if (rnd+i) % 2:
                a, b = b, a
            pairs.append((a,b))
        first_half.append(pairs)
        fixed = [fixed[0]] + [fixed[-1]] + fixed[1:-1]
    return first_half + [[(b,a) for a,b in rnd] for rnd in first_half]

def simulate_model_three(n_sims=5_000, seed=3303):
    rng = np.random.default_rng(seed)
    n = len(teams)
    historical_elo = np.array([final_states[history_name[t]]["elo"] for t in teams])
    elo = historical_elo[None,:] + 55*ordered.squad_strength.to_numpy()[None,:]
    elo = np.repeat(elo, n_sims, axis=0)
    ppg = np.array([mean_or(final_states[history_name[t]]["ppg"],1.35) for t in teams])[None,:]
    gd = np.array([mean_or(final_states[history_name[t]]["gd"],0) for t in teams])[None,:]
    home_ppg = np.array([mean_or(final_states[history_name[t]]["home_ppg"],1.55) for t in teams])[None,:]
    away_ppg = np.array([mean_or(final_states[history_name[t]]["away_ppg"],1.15) for t in teams])[None,:]
    ppg, gd = np.repeat(ppg,n_sims,axis=0), np.repeat(gd,n_sims,axis=0)
    home_ppg, away_ppg = np.repeat(home_ppg,n_sims,axis=0), np.repeat(away_ppg,n_sims,axis=0)
    regime_sd = (0.24 + 0.12*ordered.transfer_share.to_numpy()
                 + 0.08*ordered.promoted.to_numpy())
    regime = rng.standard_t(4, size=(n_sims,n))*regime_sd[None,:]
    points = np.zeros((n_sims,n), dtype=np.int16)
    class_index = {label:i for i,label in enumerate(ml_model.classes_)}

    for round_no, fixtures in enumerate(circle_schedule(n)):
        for home, away in fixtures:
            X = pd.DataFrame({
                "elo_diff":elo[:,home]-elo[:,away] + 75*(regime[:,home]-regime[:,away]),
                "recent_ppg_diff":ppg[:,home]-ppg[:,away],
                "recent_gd_diff":gd[:,home]-gd[:,away],
                "home_venue_ppg":home_ppg[:,home], "away_venue_ppg":away_ppg[:,away],
                "rest_diff":np.zeros(n_sims), "division_top":np.ones(n_sims),
                "season_progress":np.full(n_sims,round_no/38),
            })[FEATURES]
            prob = ml_model.predict_proba(X)
            u = rng.random(n_sims)
            p_a = prob[:,class_index["A"]]
            p_d = prob[:,class_index["D"]]
            away_win = u < p_a
            draw = (u >= p_a) & (u < p_a+p_d)
            home_win = ~(away_win|draw)
            hp = 3*home_win+draw; ap = 3*away_win+draw
            points[:,home] += hp; points[:,away] += ap
            margin = home_win.astype(float)-away_win.astype(float)
            ppg[:,home] = .85*ppg[:,home]+.15*hp
            ppg[:,away] = .85*ppg[:,away]+.15*ap
            gd[:,home] = .85*gd[:,home]+.15*margin
            gd[:,away] = .85*gd[:,away]-.15*margin
            home_ppg[:,home] = .88*home_ppg[:,home]+.12*hp
            away_ppg[:,away] = .88*away_ppg[:,away]+.12*ap
            expected = 1/(1+10**(-((elo[:,home]-elo[:,away]+65)/400)))
            actual = home_win + .5*draw
            change = 18*(actual-expected)
            elo[:,home] += change; elo[:,away] -= change
    positions = positions_from_points(points,rng)
    return finish_summary(points,positions,"Model 3: ML form engine"), points, positions

m3, points3, positions3 = simulate_model_three()
display(m3.sort_values("mean_points",ascending=False).style.format({
    "mean_points":"{:.1f}", "p10_points":"{:.0f}", "p90_points":"{:.0f}",
    "title_probability":"{:.1%}", "top4_probability":"{:.1%}",
    "relegation_probability":"{:.1%}", "median_position":"{:.0f}",
    "position_p10":"{:.0f}", "position_p90":"{:.0f}"}).hide(axis="index"))

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(16,7))
view=m3.sort_values("mean_points").tail(12)
xerr=np.vstack([view.mean_points-view.p10_points,view.p90_points-view.mean_points])
axes[0].errorbar(view.mean_points,view.current_team,xerr=xerr,fmt="o",color="#5D477A",ecolor="#BFB2CE",capsize=2)
axes[0].set_title("Model 3: nonlinear, path-dependent forecast")
axes[0].set_xlabel("Points, mean and 10–90% interval")

top=m3.nlargest(10,"title_probability").sort_values("title_probability")
axes[1].barh(top.current_team,top.title_probability,color="#7C63A1")
axes[1].xaxis.set_major_formatter(lambda x,_:f"{x:.0%}")
axes[1].set_title("ML engine title probability")
axes[1].set_xlabel("Probability")
plt.tight_layout();plt.show()

## Final probability table and model comparison

The **editorial consensus** is intentionally not a fourth simulation model. It is a declared weighting of the three outputs: 35% structural base, 40% contextual engine and 25% ML engine. The contextual model receives the largest weight because the August 2026 league has unusually high manager and squad churn; ML receives the smallest because five seasons is enough for useful nonlinear signals but not enough to treat the algorithm as an oracle.

In [ ]:
models={"M1 mixture + K":m1,"M2 contextual":m2,"M3 ML form":m3}
comparison=None
for label,frame in models.items():
    part=frame[["current_team","mean_points","title_probability","top4_probability","relegation_probability"]].copy()
    part.columns=["current_team"]+[f"{label} | {c}" for c in part.columns[1:]]
    comparison=part if comparison is None else comparison.merge(part,on="current_team")

weights={"M1 mixture + K":.35,"M2 contextual":.40,"M3 ML form":.25}
consensus=pd.DataFrame({"current_team":teams})
for metric in ["mean_points","title_probability","top4_probability","relegation_probability"]:
    consensus[metric]=sum(weights[label]*frame.set_index("current_team").loc[teams,metric].to_numpy()
                           for label,frame in models.items())
consensus=consensus.sort_values("mean_points",ascending=False)
comparison=comparison.merge(consensus.add_prefix("Consensus | ").rename(columns={"Consensus | current_team":"current_team"}),on="current_team")
comparison=comparison.sort_values("Consensus | mean_points",ascending=False)
comparison.to_csv(OUT/"part_two_three_model_probabilities.csv",index=False)

probability_cols=[c for c in comparison if "probability" in c]
point_cols=[c for c in comparison if "mean_points" in c]
display(comparison.style.format({**{c:"{:.1%}" for c in probability_cols},**{c:"{:.1f}" for c in point_cols}})
        .background_gradient(subset=[c for c in probability_cols if "title" in c],cmap="YlOrRd").hide(axis="index"))

In [ ]:
title_matrix=pd.DataFrame({label:frame.set_index("current_team").loc[teams,"title_probability"] for label,frame in models.items()})
title_matrix["Consensus"]=consensus.set_index("current_team").loc[teams,"title_probability"]
title_matrix=title_matrix.sort_values("Consensus",ascending=False).head(10)
plt.figure(figsize=(12,7))
sns.heatmap(title_matrix,annot=True,fmt=".1%",cmap="YlOrRd",linewidths=.5,cbar_kws={"label":"Title probability"})
plt.title("Three models, one final probability comparison")
plt.xlabel("");plt.ylabel("");plt.tight_layout();plt.show()

In [ ]:
leaders={label:frame.sort_values("title_probability",ascending=False).iloc[0] for label,frame in models.items()}
overall=consensus.iloc[0]
runner_up=consensus.iloc[1]
spread=consensus.iloc[3:9]
analysis=f'''
### Result analysis

- **Model 1:** {leaders['M1 mixture + K'].current_team} leads at {leaders['M1 mixture + K'].title_probability:.1%}. Its case is persistence: five-season performance and current squad quality are the two most stable signals. It will be strongest if 2026/27 contains no exceptional structural break.
- **Model 2:** {leaders['M2 contextual'].current_team} leads at {leaders['M2 contextual'].title_probability:.1%}. Its case is realism: stadium, coach, Europe and transfers affect different clubs differently. It should win if this high-change season is shaped by adaptation and workload.
- **Model 3:** {leaders['M3 ML form'].current_team} leads at {leaders['M3 ML form'].title_probability:.1%}. Its case is nonlinear momentum: the algorithm learns how Elo and recent form interact and updates after each simulated match. It should win if early-season feedback loops matter more than static August rankings.

### Overall prediction

The weighted view makes **{overall.current_team} the selection**, with {overall.title_probability:.1%} title probability and {overall.mean_points:.1f} expected points. **{runner_up.current_team}** is the principal alternative at {runner_up.title_probability:.1%}. The more important conclusion is that the model does not regard this as settled: the middle European race is especially open across {', '.join(spread.current_team.head(4))}.

This is a forecast, not a claim of certainty. The most defensible next improvement is walk-forward calibration of K and the contextual shock sizes against older seasons—not hand-editing probabilities until they look comfortable.
'''
display(Markdown(analysis))

## Combined evidence board

The board below brings forward four high-value Part One views, then places each Part Two engine beside its thesis and “case for correctness”. The bottom strip states the declared weighted prediction.

In [ ]:
# Re-render the key evidence into one high-resolution, presentation-ready board.
COL={"M1 mixture + K":"#315A7D","M2 contextual":"#A4672E","M3 ML form":"#6D568A"}
fig=plt.figure(figsize=(24,13),facecolor="#F6F3EC")
fig.suptitle("PREMIER LEAGUE 2026/27  •  THREE WAYS THE SEASON COULD UNFOLD",
             x=.04,y=.975,ha="left",fontsize=25,fontweight="bold",color="#17212B")
fig.text(.04,.945,"Evidence fixed at 21 August 2026  |  uncertainty is part of the result",fontsize=12,color="#53606C")

def clean(ax,title):
    ax.set_title(title,loc="left",fontsize=12,fontweight="bold",pad=10,color="#17212B")
    ax.set_facecolor("#FBFAF7")
    for s in ax.spines.values(): s.set_color("#D5D0C6")

# Four Part One graphics.
top_x=[.04,.285,.53,.775]
top_w=.20
ax=fig.add_axes([top_x[0],.63,top_w,.25]); clean(ax,"PART I · 100-player club coverage")
cnt=pl100.groupby("current_team").size().sort_values()
ax.barh(cnt.index,cnt.values,color=np.where(cnt.values==3,"#D28C45","#4E8372"));ax.axvline(3,color="#9D4C4C",ls="--",lw=1)
ax.set_xlabel("Players in constrained PL top 100");ax.tick_params(labelsize=8)

ax=fig.add_axes([top_x[1],.63,top_w,.25]); clean(ax,"PART I · Horizon changes the ranking")
rh=history_horizons.sort_values("rank_5y").set_index("current_team")[["rank_1y","rank_3y","rank_5y"]]
sns.heatmap(rh,ax=ax,cmap="YlGnBu_r",cbar=False,yticklabels=True,xticklabels=["1y","3y","5y"],linewidths=.25)
ax.tick_params(labelsize=7);ax.set_ylabel("")

ax=fig.add_axes([top_x[2],.63,top_w,.25]); clean(ax,"PART I · Current squad quality")
sq=team_inputs.sort_values("squad_strength")
ax.barh(sq.current_team,sq.squad_strength,color=np.where(sq.squad_strength>=0,"#4E8372","#D28C45"));ax.axvline(0,color="#555",lw=.8)
ax.set_xlabel("Standardised strength");ax.tick_params(labelsize=8)

ax=fig.add_axes([top_x[3],.63,top_w,.25]); clean(ax,"PART I · Context uncertainty")
uc=team_inputs.sort_values("context_sigma")
ax.barh(uc.current_team,uc.context_sigma,color="#75618E");ax.set_xlabel("Season strength σ");ax.tick_params(labelsize=8)

theses={
 "M1 mixture + K":("Persistence thesis","History + squad quality set the mean; K admits unknowns.","Best if the established hierarchy survives without a major break."),
 "M2 contextual":("Adaptation thesis","Stadium, coach, Europe and churn widen clubs differently.","Best if workload and managerial adaptation define this season."),
 "M3 ML form":("Feedback thesis","Nonlinear Elo and form update after every simulated match.","Best if early results create momentum, confidence and regime shifts."),
}
for i,(label,frame) in enumerate(models.items()):
    model_x=[.04,.36,.68][i]
    ax=fig.add_axes([model_x,.30,.28,.24]); clean(ax,label.upper())
    top=frame.nlargest(9,"mean_points").sort_values("mean_points")
    err=np.vstack([top.mean_points-top.p10_points,top.p90_points-top.mean_points])
    ax.errorbar(top.mean_points,top.current_team,xerr=err,fmt="o",color=COL[label],ecolor="#B8B5AE",capsize=2)
    ax.set_xlabel("Points: mean and 10–90% interval");ax.tick_params(labelsize=9)
    leader=frame.nlargest(1,"title_probability").iloc[0]
    fig.text(model_x,.255,f"{theses[label][0]}  ·  {leader.current_team} {leader.title_probability:.1%}",
             fontsize=11,fontweight="bold",color=COL[label])
    fig.text(model_x,.230,theses[label][1],fontsize=9,color="#25313B")
    fig.text(model_x,.207,"CASE: "+theses[label][2],fontsize=8.5,color="#5B6570")

ax=fig.add_axes([.04,.025,.92,.14]);ax.axis("off")
ax.add_patch(plt.Rectangle((0,0),1,1,transform=ax.transAxes,color="#17212B"))
top5=consensus.head(5)
winner=top5.iloc[0];second=top5.iloc[1]
ax.text(.025,.74,"OVERALL CALL",transform=ax.transAxes,color="#D9B56D",fontsize=12,fontweight="bold")
ax.text(.025,.43,f"{winner.current_team} — {winner.title_probability:.1%}",transform=ax.transAxes,color="white",fontsize=26,fontweight="bold")
ax.text(.025,.19,f"{winner.mean_points:.1f} expected points  •  main alternative: {second.current_team} {second.title_probability:.1%}",
        transform=ax.transAxes,color="#D4DBE1",fontsize=12)
ax.text(.42,.69,"WHY",transform=ax.transAxes,color="#D9B56D",fontsize=11,fontweight="bold")
ax.text(.42,.45,"The structural evidence is strong, but the contextual and ML engines\nkeep meaningful routes open for rivals. The prediction is a weighted\njudgement—not an average designed to hide disagreement.",
        transform=ax.transAxes,color="white",fontsize=12,va="center")
ax.text(.77,.69,"MODEL WEIGHTS",transform=ax.transAxes,color="#D9B56D",fontsize=11,fontweight="bold")
ax.text(.77,.43,"35%  mixture + K\n40%  contextual\n25%  ML form",transform=ax.transAxes,color="white",fontsize=12,va="center")

board=OUT/"premier_league_part_two_evidence_board.png"
plt.savefig(board,dpi=160,bbox_inches="tight",facecolor=fig.get_facecolor())
plt.show()
print("Saved",board)

### Source and calibration note

Player priors use [EA SPORTS FC 26 ratings](https://www.ea.com/games/ea-sports-fc/ratings?gender=0&orderBy=rank&page=1) and the [Fantasy Premier League snapshot](https://fantasy.premierleague.com/api/bootstrap-static/). Historical training results come from [football-data.co.uk](https://www.football-data.co.uk/englandm.php). Manager and European-competition context follows the [Premier League manager guide](https://www.premierleague.com/en/news/4679012/meet-the-202627-premier-league-clubs-managers) and [2026/27 key dates/qualification summary](https://www.premierleague.com/en/news/4671514/summer-2026-key-football-dates-for-your-calendar).

The ML validation reports discrimination, not proof of calibrated 2026/27 title probabilities. All three engines still require multi-season walk-forward probability calibration before decision-grade use.